In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import numpy as np
import torch
import pprint

# add parent dir so importing top level files works in notebook subdir
parent_dir = str(Path().resolve().parent)
sys.path.insert(0, parent_dir)

from llm import qwen_utils, tools
from autoencoder.model_qwen import QwenAutoencoder

/home/guests/nicolas_stellwag/surgery-scene-graphs/.pixi/envs/default/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
graph_dir = Path("output/qwen3_lesstricks/video01_00240/graph")
ae_path = Path("data/preprocessed/qwen3/video01_00240/autoencoder_me/best_ckpt.pth")

print(f"Graph dir: {graph_dir}")
print(f"Autoencoder path: {ae_path}")

Graph dir: output/qwen3_lesstricks/video01_00240/graph
Autoencoder path: data/preprocessed/qwen3/video01_00240/autoencoder_me/best_ckpt.pth


In [14]:
# Load autoencoder
autoencoder = QwenAutoencoder(input_dim=4096 * 4, latent_dim=3).to("cuda")
autoencoder.load_state_dict(torch.load(ae_path, map_location="cuda"))
autoencoder.eval()
print(f"Autoencoder loaded from {ae_path}")

# Load graph data and create toolkit (accepts npz directly)
toolkit = tools.GraphTools(
    positions=np.load(graph_dir / "positions.npy"),
    clusters=np.load(graph_dir / "clusters.npy"),
    centroids=np.load(graph_dir / "c_centroids.npy"),
    centers=np.load(graph_dir / "c_centers.npy"),
    extents=np.load(graph_dir / "c_extents.npy"),
    adjacency=np.load(graph_dir / "graph.npy"),
    bhattacharyya_coeffs=np.load(graph_dir / "bhattacharyya_coeffs.npy"),
    qwen_feats=np.load(graph_dir / "c_qwen_feats.npz"),
    patch_latents_through_time=np.load(graph_dir / "patch_latents_through_time.npy"),
    autoencoder=autoencoder,
)
graph_tools = toolkit.get_all_tools()
print(f"Loaded {len(graph_tools)} tools: {list(graph_tools.keys())}")
print(f"Graph has {len(np.unique(toolkit.clusters))} nodes, {toolkit.adjacency.shape[0]} timesteps")

Autoencoder loaded from data/preprocessed/qwen3/video01_00240/autoencoder_me/best_ckpt.pth
Loaded 7 tools: ['node_distances_through_time', 'node_overlap_scores_through_time', 'node_overlap_position_at_time', 'inspect_highres_node_at_time', 'inspect_node_through_time', 'inspect_scene_at_time', 'sample_spatial_grid_at_time']
Graph has 11 nodes, 20 timesteps


In [4]:
# Load patched Qwen3-VL model with custom feature injection support
model, processor = qwen_utils.get_patched_qwen3()
print(f"Model loaded on device: {model.device}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded on device: cuda:0


In [16]:
response = qwen_utils.prompt_graph_agent(
    question="Use the patch grid tool iteratively to locate where the grasper touches the gallbladder at timestep 10 as accurately as possible.",
    node_feats=toolkit.qwen_feats,
    initial_timestep_idx=0,
    node_centers=toolkit.point_o2n(toolkit.centers),
    node_centroids=toolkit.point_o2n(toolkit.centroids),
    node_extents=toolkit.distance_o2n(toolkit.extents),
    model=model,
    processor=processor,
    qwen_version="qwen3",
    system_prompt="You are a surgical scene understanding assistant analyzing a brief excerpt of a cholecystectomy procedure. The procedure is represented as a 4D spatiotemporal graph, of which you receive a rough description at timestep 0 (in (0..19)), which can be further explored with your tools. Answer user questions by reasoning over which tools to use first, exploring the graph until you are confident about your answer, and only then reporting your final answer.",
    tools=graph_tools,
)
if type(response) == dict:
    pprint.pprint(response["message_history"])
else:
    print(response)

[{'content': [{'text': 'You are a surgical scene understanding assistant '
                       'analyzing a brief excerpt of a cholecystectomy '
                       'procedure. The procedure is represented as a 4D '
                       'spatiotemporal graph, of which you receive a rough '
                       'description at timestep 0 (in (0..19)), which can be '
                       'further explored with your tools. Answer user '
                       'questions by reasoning over which tools to use first, '
                       'exploring the graph until you are confident about your '
                       'answer, and only then reporting your final answer.',
               'type': 'text'}],
  'role': 'system'},
 {'content': [{'text': '<graph-nodes t="0">\n', 'type': 'text'},
              {'text': '<node id="0">\n', 'type': 'text'},
              {'text': '<lowres-visual-descriptor>', 'type': 'text'},
              {'image': None, 'type': 'image'},
              {'

In [18]:
for c in response["tool_calls"]:
    print(c["arguments"])

{'timestep': 10}
{'timestep': 10, 'grid_size': 5}
{'timestep': 10, 'grid_size': 5, 'bbox': [0.1, 0.5, -0.5, 0.5, 0.5, 1.0]}
